# Label
This notebook is used for labelling the bibtex data with the reference data, scope:
- Labelling manually 5 papers, each of these must ensure to have at least 20 rows in bibtex table
- Labelling automatically 500 papers in the remaining pool of bibtex table entries, which exclude manually labled rows.

The expected output of this notebook is a csv file containing ground truth of labelled data.

# Setup
This section is used to import necessary library.

In [32]:
from pathlib import Path
import pandas as pd
import numpy as np
from manual import ground_truth_manual

from rapidfuzz import fuzz

DIR_DATA = '../scopus/'
SAVE_DATA = '../dataset/'

# I. Exploration of scopus (bibtex and references table)

### Read data from scopus
- Read data of references table and bibtex table
- The bibtex table is splitted into shard for smaller size so it need to be concatenated when loaded again by pandas

In [2]:
files = sorted(Path(DIR_DATA + 'bibtex').glob("*.parquet"))

references_df = pd.read_parquet(DIR_DATA + 'references.parquet')
bibtex_df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

### Info lookup
- Explore the information of 2 tables

In [3]:
references_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81795 entries, 0 to 81794
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   arxiv_id             81795 non-null  str   
 1   ref_arxiv_id         81795 non-null  str   
 2   ref_title            81795 non-null  str   
 3   ref_authors          81795 non-null  object
 4   ref_submission_date  81651 non-null  str   
 5   ref_s2_id            81795 non-null  str   
dtypes: object(1), str(5)
memory usage: 14.8+ MB


In [4]:
bibtex_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 181344 entries, 0 to 181343
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   arxiv_id      181344 non-null  str     
 1   citation_key  181344 non-null  str     
 2   entry_type    181344 non-null  category
 3   title         177129 non-null  str     
 4   authors       181344 non-null  object  
 5   year          180171 non-null  float64 
 6   doi           71144 non-null   str     
 7   journal       149277 non-null  str     
 8   url           41541 non-null   str     
dtypes: category(1), float64(1), object(1), str(6)
memory usage: 36.0+ MB


### Row counting for each arXiv ID 
- With each arXiv ID, counting its number of rows in bibtex table

In [5]:
arXiv_row_count_bibtex = bibtex_df['arxiv_id'].value_counts()
arXiv_row_count_bibtex

arxiv_id
2608.07247    620
2608.07221    608
2608.05179    497
2608.06978    489
2608.05885    345
             ... 
2608.06884      2
2608.07737      2
2608.07842      2
2608.08251      2
2608.08458      1
Name: count, Length: 4274, dtype: int64

- With each arXiv ID, counting its number of rows in references table

In [6]:
arXiv_row_count_references = references_df['arxiv_id'].value_counts()
arXiv_row_count_references

arxiv_id
2608.07221    447
2608.07247    427
2608.05885    310
2608.07514    191
2608.05184    185
             ... 
2608.08845      1
2608.08933      1
2608.08934      1
2608.08993      1
2608.08997      1
Name: count, Length: 4288, dtype: int64

### Filter rows that are appropriate with requirements
- We only those arXiv IDs which have the number of rows is equal or greater than 20
- I gonna choose the upper bound as 30 for easily manual label :))

In [7]:
arXiv_filter_bibtex = arXiv_row_count_bibtex[(arXiv_row_count_bibtex >= 20) & (arXiv_row_count_bibtex <= 30)].index
len(arXiv_filter_bibtex)

869

- We also do the same thing with references table
- This is for effective and sufficient labelling later

In [8]:
arXiv_filter_references = arXiv_row_count_references[(arXiv_row_count_references >= 20) & (arXiv_row_count_references <= 30)].index
len(arXiv_filter_references)

810

- Besides the appropriate row quantity, the arXiv IDs must also be existent in references table

In [9]:
arXiv_filter = arXiv_filter_bibtex[arXiv_filter_bibtex.isin(arXiv_filter_references)]
len(arXiv_filter)

97

- Oh, there are only 97 appropriate arXiv IDs for manual labelling.

In [37]:
bibtex_filter_auto = arXiv_row_count_bibtex[arXiv_row_count_bibtex >= 2].index
ref_filter_auto = arXiv_row_count_references[arXiv_row_count_references >= 2].index
arXiv_filter_auto = bibtex_filter_auto[bibtex_filter_auto.isin(ref_filter_auto)]
len(arXiv_filter_auto)

3555

# II. Manual Labelling

### Choosing samples
- I randomly choose 5 five samples from filter arXiv ID set

In [ ]:
rng = np.random.default_rng(42)
selected = rng.choice(arXiv_filter, size=5, replace=False)
selected

array(['2608.06628', '2608.06434', '2608.04791', '2608.06791',
       '2608.05850'], dtype=object)

- I have chosen 5 arXiv IDs from the filter set (this ensures the selected publications to have at least 20 rows in bibtex table)
- Let see the number of rows for each of these arXiv ID in bibtex table.

In [11]:
arXiv_row_count_bibtex[selected]

arxiv_id
2608.06628    25
2608.06434    28
2608.04791    26
2608.06791    30
2608.05850    20
Name: count, dtype: int64

- Firstly, we must ensure those selected arXiv IDs to exist in references dataframe (these are filtered above, but we should perform this again to check)

In [12]:
mask_sum = np.isin(selected,arXiv_row_count_references.index).sum()
mask_sum == len(selected)

np.True_

- Good, all arXiv IDs that we have selected exist in references table
- Let see the number of rows corresponding with these arXiv IDs in references table

In [13]:
arXiv_row_count_references[selected]

arxiv_id
2608.06628    22
2608.06434    27
2608.04791    22
2608.06791    22
2608.05850    21
Name: count, dtype: int64

- Let see the first publication for manual labelling this

In [ ]:
bibtex_rows = bibtex_df.loc[bibtex_df['arxiv_id'] == selected[4]]
ref_rows = references_df.loc[references_df['arxiv_id'] == selected[4]]

In [ ]:
bibtex_rows

,arxiv_id,citation_key,entry_type,title,authors,year,doi,journal,url
68819,2608.05850,nllb-24,article,Scaling neural machine translation to 200 lang...,"[NLLB Team, Marta R. Costa-jussà, James Cross,...",2024.0,10.1038/s41586-024-07335-x,Nature,https://doi.org/10.1038/s41586-024-07335-x
68820,2608.05850,sentalign-2023,inproceedings,SentAlign: Accurate and Scalable Sentence Alig...,"[Steinþór Steingrímsson, Hrafn Loftsson, Andy ...",2023.0,NaN,Proceedings of the 2023 Conference on Empirica...,NaN
68821,2608.05850,wu2025bitterlessonlearned2000,misc,"The Bitter Lesson Learned from 2,000+ Multilin...","[Minghao Wu, Weixuan Wang, Sinuo Liu, Huifeng ...",2025.0,NaN,NaN,https://arxiv.org/abs/2504.15521
68822,2608.05850,urieli2025jochre,article,Jochre 3 and the Yiddish OCR corpus,"[Assaf Urieli, Amber Clooney, Michelle Sigiel,...",2025.0,NaN,arXiv preprint arXiv:2501.08442,NaN
68823,2608.05850,yiddishUD,misc,YiTB: the Yiddish Tree Bank,[Matthew Kirk Andrews],2025.0,NaN,NaN,https://github.com/UniversalDependencies/UD_Yi...
68824,2608.05850,berkovitch_rusinek_yiddish_2022,misc,New Languages for NLP: Yiddish — Annotated Dat...,"[Ephraim Berkovitch, Sinai Rusinek]",2021.0,NaN,NaN,https://github.com/New-Languages-for-NLP/yiddish
68825,2608.05850,koto2025sherkalachat,inproceedings,Sherkala-Chat: Building a State-of-the-Art LLM...,"[Fajri Koto, Rituraj Joshi, Nurdaulet Mukhitul...",2025.0,NaN,Second Conference on Language Modeling,https://openreview.net/forum?id=wRcTCcb0H5
68826,2608.05850,bleaman2025corpus,article,"The corpus of spoken Yiddish in Europe: Goals,...","[Isaac L Bleaman, Chaya R Nove]",2025.0,NaN,Language Documentation & Conservation,NaN
68827,2608.05850,webber2022reyd,inproceedings,REYD-The First Yiddish Text-to-Speech Dataset ...,"[Jacob Webber, Samuel K Lo, Isaac L Bleaman]",2022.0,NaN,INTERSPEECH,NaN
68828,2608.05850,kulick2022part,article,A part-of-speech tagger for Yiddish,"[Seth Kulick, Neville Ryant, Beatrice Santorin...",2022.0,NaN,arXiv preprint arXiv:2204.01175,NaN


In [ ]:
ref_rows

,arxiv_id,ref_arxiv_id,ref_title,ref_authors,ref_submission_date,ref_s2_id
30648,2608.05850,2506.00288,Emergent Abilities of Large Language Models un...,"[Ahmed Elhady, Eneko Agirre, Mikel Artetxe]",2025-05-30,e943d1ac1e74d4290a7bf8e16d01394d964220ef
30649,2608.05850,2505.22232,Judging Quality Across Languages: A Multilingu...,"[Mehdi Ali, Manuel Brack, Max Lübbering, Elias...",2025-05-28,5df67582cd7ed394c0180ca2e3ab59ce7c0516a8
30650,2608.05850,2505.09388,Qwen3 Technical Report,"[An Yang, An-Feng Li, Bao-Song Yang, Bei-Chen ...",2025-05-14,d2d84d56f730f81d276a02b48d5d44db5bde0b4a
30651,2608.05850,2504.15521,"The Bitter Lesson Learned from 2,000+ Multilin...","[Minghao Wu, Wei-Xuan Wang, Sinuo Liu, Huifeng...",2025-04-22,4809f38b9162ae3609a44129c8fef6d8a8d9ecaf
30652,2608.05850,2503.01493,Sherkala-Chat: Building a State-of-the-Art LLM...,"[Fajri Koto, Rituraj Joshi, Nurdaulet Mukhitul...",2025-03-03,2ddcbb6189731a8e4e088dff15ae3f333b7752e6
30653,2608.05850,2501.08442,Jochre 3 and the Yiddish OCR corpus,"[A. Urieli, Amber Clooney, Michelle Sigiel, Gr...",2025-01-14,89ef57c269ea3977862be9bc34c6a2ad940a45c2
30654,2608.05850,2408.02239,Pula: Training Large Language Models for Setswana,"[Nathan Brown, V. Marivate]",2024-08-05,0dc574a4d9788285a8d051bf0a7104c7584bb3ec
30655,2608.05850,2407.21783,The Llama 3 Herd of Models,"[Aaron Grattafiori, Abhimanyu Dubey, Abhinav J...",2024-07-31,9da8ea6466d7142703d8771d8dcea44e19304abb
30656,2608.05850,2404.04042,Teaching Llama a New Language Through Cross-Li...,"[Hele-Andra Kuulmets, Taido Purason, Agnes Luh...",2024-04-05,8dc4ac4446f787f0647e1e16a9c43b2880e04008
30657,2608.05850,2403.20266,Latxa: An Open Language Model and Evaluation S...,"[Julen Etxaniz, Oscar Sainz, Naiara Pérez, Itz...",2024-03-29,7cccc78f24e4eeca5356151b3d3557c1f45b6294


- Manual labelling with 5 publications

In [18]:
ground_truth_manual.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   arxiv_id      102 non-null    str  
 1   citation_key  102 non-null    str  
 2   ref_arxiv_id  102 non-null    str  
dtypes: str(3)
memory usage: 5.9 KB


In [19]:
ground_truth_manual.head()

,arxiv_id,citation_key,ref_arxiv_id
0,2608.06628,kim2025train,2502.06768
1,2608.06628,ye2024beyond,2410.14157
2,2608.06628,song2025seed,2508.02193
3,2608.06628,trainin2026discrete,2602.19980
4,2608.06628,schiff2025simple,2412.10193


- Let reorganize the ground truth

In [23]:
truth = ground_truth_manual.rename(
    columns={"ref_arxiv_id": "true_arxiv_id"}
)

candidates = (
    references_df[["arxiv_id", "ref_arxiv_id"]]
    .rename(columns={"ref_arxiv_id": "candidate_arxiv_id"})
)

manual_labeled = truth.merge(candidates, on="arxiv_id", how="inner")

manual_labeled["label"] = (
    manual_labeled["candidate_arxiv_id"] == manual_labeled["true_arxiv_id"]
).astype(int)

manual_labeled = manual_labeled[["arxiv_id", "citation_key", "candidate_arxiv_id", "label"]]

manual_labeled.info()

<class 'pandas.DataFrame'>
RangeIndex: 2370 entries, 0 to 2369
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   arxiv_id            2370 non-null   str  
 1   citation_key        2370 non-null   str  
 2   candidate_arxiv_id  2370 non-null   str  
 3   label               2370 non-null   int64
dtypes: int64(1), str(3)
memory usage: 152.8 KB


- Let save the manual ground truth into a csv file

In [26]:
manual_labeled.to_csv(SAVE_DATA + 'manual_truth.csv', index=False)

## III. Automatical Labelling

- Firstly, I gonna choose 500 publications in the remaining pool, excluding those are manual labelled
- There is no requirement for the number of rows for these automatical publications

In [38]:
remaining_pool = arXiv_filter_auto[~arXiv_filter_auto.isin(selected)]

rng = np.random.default_rng(42)
selected_auto = rng.choice(remaining_pool, size=500, replace=False)
selected_auto[:5]

array(['2608.06665', '2608.07933', '2608.06638', '2608.04497',
       '2608.05903'], dtype=object)

- The algorithm for automatical labelling is fuzz score
- It states that the score reflects the fit rate between two entries. The higher the score is, the higher the fit rate is 

In [48]:
MARGIN = 10
THRESHOLD = 95

auto_truth = []
for paper_id in selected_auto:
    bib_rows = bibtex_df.loc[bibtex_df['arxiv_id'] == paper_id]
    ref_rows= references_df.loc[references_df['arxiv_id'] == paper_id]

    for _, bib_entry in bib_rows.iterrows():
        scores = []
        if not isinstance(bib_entry['title'], str) or not bib_entry['title'].strip():
            continue 
        for _, candidate in ref_rows.iterrows():
            score = fuzz.ratio(
                bib_entry['title'].lower().strip(),
                candidate['ref_title'].lower().strip()
            )
            scores.append(score)
        scores = np.array(scores)
        second_idx, best_idx = np.argsort(scores)[-2:]
        best_score, second_score = scores[best_idx], scores[second_idx]
        accept = (
            (best_score >= THRESHOLD)
            and (best_score - second_score >= MARGIN)
        )
        print (f"Paper ID: {paper_id}, Bib Entry: {bib_entry['citation_key']}, Best Score: {best_score}, Second Score: {second_score}, Accept: {accept}")
        print(f'Best candidate: {ref_rows.iloc[best_idx]["ref_title"]}, Second candidate: {ref_rows.iloc[second_idx]["ref_title"]}')
        print(f'Bibtex title: {bib_entry["title"]}')
        print()
        if accept:
            auto_truth.append({
                'arxiv_id': bib_entry['arxiv_id'],
                'citation_key': bib_entry['citation_key'],
                'ref_arxiv_id': ref_rows.iloc[best_idx]['ref_arxiv_id'],
            })

Paper ID: 2608.06665, Bib Entry: mcclain2021behaviors, Best Score: 46.15384615384615, Second Score: 41.9047619047619, Accept: False
Best candidate: Emerging Methods of Auction Design in Social Networks, Second candidate: Graph evolution: Densification and shrinking diameters
Bibtex title: The behaviors and attitudes of US adults on Twitter

Paper ID: 2608.06665, Bib Entry: leicht2006vertex, Best Score: 98.30508474576271, Second Score: 45.714285714285715, Accept: True
Best candidate: Vertex similarity in networks., Second candidate: Statistical mechanics of complex networks
Bibtex title: Vertex similarity in networks

Paper ID: 2608.06665, Bib Entry: ilprints422, Best Score: 46.846846846846844, Second Score: 40.0, Accept: False
Best candidate: Graph evolution: Densification and shrinking diameters, Second candidate: Emerging Methods of Auction Design in Social Networks
Bibtex title: The PageRank Citation Ranking: Bringing Order to the Web.

Paper ID: 2608.06665, Bib Entry: aggarwal2019a

In [56]:
ground_truth_auto = pd.DataFrame(auto_truth)
ground_truth_auto.info()

<class 'pandas.DataFrame'>
RangeIndex: 9156 entries, 0 to 9155
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   arxiv_id      9156 non-null   str  
 1   citation_key  9156 non-null   str  
 2   ref_arxiv_id  9156 non-null   str  
dtypes: str(3)
memory usage: 530.9 KB


In [57]:
ground_truth_auto.head()

,arxiv_id,citation_key,ref_arxiv_id
0,2608.06665,leicht2006vertex,physics/0510143
1,2608.06665,zhang2021vigdet,2110.15454
2,2608.06665,sheth2022defining,2104.10788
3,2608.06665,garg2023handling,2202.00126
4,2608.06665,jhaver2023personalizing,2305.10374


- Check the automatical labelling set whether it has duplicate rows (taking arxiv_id and ref_arxiv_id column into consideration).

In [54]:
ground_truth_auto. duplicated(subset=['arxiv_id', 'ref_arxiv_id']).sum()

mask = ground_truth_auto.duplicated(subset=['arxiv_id', 'ref_arxiv_id'], keep=False)
ground_truth_auto.loc[mask]

,arxiv_id,citation_key,ref_arxiv_id
59,2608.04497,ALPSCore1,1609.03930
60,2608.04497,ALPSCore2,1609.03930
871,2608.06772,Emami2018,2307.00142
880,2608.06772,Emami2023BuildingsBench,2307.00142
6051,2608.07778,leung_nomoto_2018,2007.08466
6054,2608.07778,leung_nomoto_2020a,2007.08466
6055,2608.07778,leung_nomoto_2020b,2007.08466
6857,2608.07209,scolnic2018complete,1710.00845
6865,2608.07209,Pan-STARRS1:2017jku,1710.00845


- Oh, there are some duplicated rows here
- I gonna check them whether they have the same title

In [ ]:
mask_df = ground_truth_auto[mask][['arxiv_id', 'citation_key']]

matched_bibtex = bibtex_df.merge(
    mask_df,
    on=["arxiv_id", "citation_key"],
    how="inner"
)

matched_bibtex

,arxiv_id,citation_key,entry_type,title,authors,year,doi,journal,url
0,2608.04497,ALPSCore1,article,Updated core libraries of the ALPS project,"[A. Gaenko, A. E. Antipov, G. Carcassi, T. Che...",2017.0,10.1016/j.cpc.2016.12.009,Computer Physics Communications,https://www.sciencedirect.com/science/article/...
1,2608.04497,ALPSCore2,techreport,Updated Core Libraries of the ALPS Project,"[Markus Wallerberger, Sergei Iskakov, Alexande...",2018.0,10.48550/arxiv.1811.08331,NaN,http://arxiv.org/abs/1811.08331
2,2608.06772,Emami2018,misc,BuildingsBench: A Large-Scale Dataset of 900K ...,"[Patrick Emami, Peter Graf]",2018.0,10.25984/1986147,NaN,NaN
3,2608.06772,Emami2023BuildingsBench,inproceedings,BuildingsBench: A Large-Scale Dataset of 900K ...,"[Patrick Emami, Abhijeet Sahu, Peter Graf]",2023.0,10.52202/075280-0871,Advances in Neural Information Processing Syst...,NaN
4,2608.07209,scolnic2018complete,article,The complete light-curve sample of spectroscop...,"[D. M. Scolnic, D. Jones, A. Rest, Y. Pan, R. ...",2018.0,NaN,The Astrophysical Journal,NaN
5,2608.07209,Pan-STARRS1:2017jku,article,The Complete Light-curve Sample of Spectroscop...,"[Pan-STARRS1 Collaboration, D. M. Scolnic]",2018.0,NaN,Astrophys. J,http://www.arXiv.org/abs/1710.00845
6,2608.07778,leung_nomoto_2018,article,Explosive Nucleosynthesis in Near-Chandrasekha...,"[Shing-Chi Leung, Ken'ichi Nomoto]",2018.0,10.3847/1538-4357/aac2df,NaN,NaN
7,2608.07778,leung_nomoto_2020a,article,Explosive Nucleosynthesis in Sub-Chandrasekhar...,"[Shing-Chi Leung, Ken'ichi Nomoto]",2020.0,10.3847/1538-4357/ab5c1f,NaN,NaN
8,2608.07778,leung_nomoto_2020b,article,Explosive Nucleosynthesis in Near-Chandrasekha...,"[Shing-Chi Leung, Ken'ichi Nomoto]",2020.0,10.3847/1538-4357/aba1e3,NaN,NaN


- These duplicated rows have the same title in their own publication, so I gonna keep them in the ground truth
- I gonna do the same thing like the manual label stage: reorganize the ground truth dataframe and then save it into a csv file

In [ ]:
truth_auto = ground_truth_auto.rename(
    columns={"ref_arxiv_id": "true_arxiv_id"}
)

candidates = (
    references_df[["arxiv_id", "ref_arxiv_id"]]
    .rename(columns={"ref_arxiv_id": "candidate_arxiv_id"})
)

auto_labeled = truth_auto.merge(candidates, on="arxiv_id", how="inner")

auto_labeled["label"] = (
    auto_labeled["candidate_arxiv_id"] == auto_labeled["true_arxiv_id"]
).astype(int)

auto_labeled = auto_labeled[["arxiv_id", "citation_key", "candidate_arxiv_id", "label"]]

auto_labeled.info()